# Analytics Algorithms

Dual anomaly detection (z-score + IQR), severity classification, and Holt-Winters forecasting.

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List, Optional

DATA_PATH = "data/processed"

monthly = pd.read_parquet(f"{DATA_PATH}/monthly_metrics.parquet")
monthly["month"] = pd.to_datetime(monthly["month"])
monthly = monthly.sort_values("month").reset_index(drop=True)

print(f"Loaded {len(monthly)} months: {monthly['month'].min().strftime('%Y-%m')} to {monthly['month'].max().strftime('%Y-%m')}")

## 1. Z-Score Anomaly Detection

Deseasonalization is implicit: z-score operates on the raw series mean/std. For a 24-month SaaS series with no strong seasonality, this is sufficient. Rolling mean deseasonalization would require at least 3 full seasonal cycles.

In [ ]:
def anomaly_severity(zscore: float) -> str:
    """Classify anomaly severity based on absolute z-score.

    Thresholds follow the empirical rule:
        |z| > 3: outside 99.7% of a normal distribution -- critical
        |z| > 2: outside 95.4% -- warning
        |z| > 1.5: outside 86.6% -- info (used as lower bound for flagging)
    """
    abs_z = abs(zscore)
    if abs_z > 3:
        return "critical"
    elif abs_z > 2:
        return "warning"
    return "info"


def detect_anomalies_zscore(
    series: pd.Series,
    index_labels: Optional[pd.Series] = None,
    threshold: float = 2.0,  # sigma threshold: 2.0 catches ~5% of points under normality
) -> List[Dict]:
    if series.empty or series.std() == 0:
        return []

    mean = series.mean()
    std = series.std()
    zscores = (series - mean) / std

    anomalies = []
    for i, z in enumerate(zscores):
        if abs(z) >= threshold:
            label = str(index_labels.iloc[i]) if index_labels is not None else str(i)
            anomalies.append({
                "month": label,
                "value": round(float(series.iloc[i]), 4),
                "zscore": round(float(z), 4),
                "severity": anomaly_severity(z),
            })
    return anomalies


# Detect anomalies in MRR
labels = monthly["month"].dt.strftime("%Y-%m")
mrr_anomalies = detect_anomalies_zscore(monthly["mrr"], index_labels=labels)

print(f"MRR anomalies detected: {len(mrr_anomalies)}")
for a in mrr_anomalies:
    print(f"  {a['month']}  value={a['value']:>10,.0f}  z={a['zscore']:>+.2f}  {a['severity']}")

## 2. IQR Anomaly Detection

IQR is non-parametric and does not assume normality. It outperforms z-score when the metric distribution has heavy tails (e.g., CAC, which can spike dramatically).

In [ ]:
def detect_anomalies_iqr(
    series: pd.Series,
    index_labels: Optional[pd.Series] = None,
    multiplier: float = 1.5,  # Tukey's fences: 1.5 is standard; 3.0 for "far outliers"
) -> List[Dict]:
    if series.empty:
        return []

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    if iqr == 0:  # constant series -- no outliers by definition
        return []

    lower_fence = q1 - multiplier * iqr
    upper_fence = q3 + multiplier * iqr

    mean = series.mean()
    std = series.std() if series.std() > 0 else 1.0

    anomalies = []
    for i, val in enumerate(series):
        if val < lower_fence or val > upper_fence:
            # Approximate z-score for severity classification (method is IQR, but severity scale stays consistent)
            approx_z = (val - mean) / std
            label = str(index_labels.iloc[i]) if index_labels is not None else str(i)
            anomalies.append({
                "month": label,
                "value": round(float(val), 4),
                "zscore": round(float(approx_z), 4),
                "severity": anomaly_severity(approx_z),
            })
    return anomalies


# Compare z-score vs IQR on churn (which can spike suddenly)
churn_z = detect_anomalies_zscore(monthly["logo_churn_rate"], index_labels=labels)
churn_iqr = detect_anomalies_iqr(monthly["logo_churn_rate"], index_labels=labels)

print(f"Churn anomalies -- z-score: {len(churn_z)}, IQR: {len(churn_iqr)}")

q1 = monthly["logo_churn_rate"].quantile(0.25)
q3 = monthly["logo_churn_rate"].quantile(0.75)
iqr = q3 - q1
print(f"\nIQR fences: lower={q1 - 1.5*iqr:.4f}, upper={q3 + 1.5*iqr:.4f}")
print(f"z-score bounds: +/-2 sigma = [{monthly['logo_churn_rate'].mean() - 2*monthly['logo_churn_rate'].std():.4f}, {monthly['logo_churn_rate'].mean() + 2*monthly['logo_churn_rate'].std():.4f}]")

## 3. Dual-Method Rationale

Using both methods reduces false negatives: a data point only needs to be flagged by one method. Severity is taken as the maximum, since both methods agree that the point is unusual -- the stricter estimate is more conservative.

In [ ]:
_SEVERITY_RANK = {"critical": 3, "warning": 2, "info": 1}


def detect_anomalies_dual(
    series: pd.Series,
    index_labels: Optional[pd.Series] = None,
    zscore_threshold: float = 2.0,
    iqr_multiplier: float = 1.5,
) -> List[Dict]:
    """Union of z-score and IQR detections; severity = max of both methods."""
    z_results = {a["month"]: a for a in detect_anomalies_zscore(series, index_labels, zscore_threshold)}
    iqr_results = {a["month"]: a for a in detect_anomalies_iqr(series, index_labels, iqr_multiplier)}

    all_months = set(z_results.keys()) | set(iqr_results.keys())
    combined = []

    for month in sorted(all_months):
        z_entry = z_results.get(month)
        iqr_entry = iqr_results.get(month)
        base = z_entry or iqr_entry

        if z_entry and iqr_entry:
            # Both methods agree -- take max severity
            sev = max(z_entry["severity"], iqr_entry["severity"],
                      key=lambda s: _SEVERITY_RANK[s])
        else:
            sev = base["severity"]

        combined.append({
            "month": month,
            "value": base["value"],
            "zscore": base["zscore"],
            "severity": sev,
            "detected_by": (
                "both" if (z_entry and iqr_entry)
                else ("zscore" if z_entry else "iqr")
            ),
        })

    return combined


# Run dual detection across all key metrics
metrics_to_check = ["mrr", "logo_churn_rate", "nps", "nrr", "cac"]
all_anomalies = []

for col in metrics_to_check:
    if col in monthly.columns:
        anoms = detect_anomalies_dual(monthly[col].astype(float), index_labels=labels)
        for a in anoms:
            a["metric"] = col
        all_anomalies.extend(anoms)

print(f"Total anomalies across all metrics: {len(all_anomalies)}")
critical = [a for a in all_anomalies if a["severity"] == "critical"]
warnings = [a for a in all_anomalies if a["severity"] == "warning"]
print(f"  Critical: {len(critical)}  Warning: {len(warnings)}  Info: {len(all_anomalies)-len(critical)-len(warnings)}")

for a in sorted(all_anomalies, key=lambda x: _SEVERITY_RANK[x["severity"]], reverse=True)[:5]:
    print(f"  {a['metric']:<22} {a['month']}  {a['severity']:<8}  detected_by={a['detected_by']}")

## 4. Severity Classification Summary

In [ ]:
import pandas as pd

severity_table = pd.DataFrame([
    {"level": "critical", "z_threshold": "|z| > 3",   "probability": "0.27% under normality",  "action": "Immediate investigation"},
    {"level": "warning",  "z_threshold": "|z| > 2",   "probability": "4.55% under normality",  "action": "Monitor closely"},
    {"level": "info",     "z_threshold": "|z| > 1.5", "probability": "13.36% under normality", "action": "Log and track"},
])

print(severity_table.to_string(index=False))

## 5. Holt-Winters Forecasting

Additive trend is chosen over multiplicative because the MRR series grows roughly linearly, not exponentially -- no strong multiplicative seasonal effect in 24 months.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

mrr_series = monthly["mrr"].astype(float).values

model = ExponentialSmoothing(
    mrr_series,
    trend="add",       # additive: level + trend*t (linear growth)
    seasonal=None,     # no seasonal component: 24 months insufficient for annual seasonality estimation
    initialization_method="estimated",  # MLE initialization vs heuristic "simple"
).fit(
    optimized=True     # scipy.optimize finds alpha/beta that minimize SSE on training data
)

periods = 6
forecast = model.forecast(periods)
fitted = model.fittedvalues

print(f"Model parameters: alpha={model.params['smoothing_level']:.3f}, beta={model.params.get('smoothing_trend', 0):.3f}")
print(f"\n6-month MRR forecast:")
for i, v in enumerate(forecast, 1):
    print(f"  Period +{i}: ${v:>10,.0f}")

## 6. Forecast Confidence Intervals

Bootstrap residuals rather than analytical intervals: with 24 data points, normality of residuals cannot be assumed. 80% CI chosen over 95% to reflect honest uncertainty about a short series.

In [ ]:
# Residual standard error from in-sample fit
residuals = mrr_series - fitted
rse = float(np.std(residuals))

# CI multiplier: 1.28 = 80% CI, 1.645 = 90%, 1.96 = 95%
# Using 1.96 (95%) in production to be conservative, but interval widens with horizon.
ci_mult = 1.96

print(f"Residual standard error: ${rse:,.0f}")
print(f"\nForecast with CI (widening by sqrt(1 + 0.1*i) per period):")
print(f"{'Period':<8} {'Forecast':>12} {'CI Lower':>12} {'CI Upper':>12} {'Width':>12}")
print("-" * 56)

for i, v in enumerate(forecast):
    # CI width grows with forecast horizon: uncertainty compounds
    width = rse * ci_mult * np.sqrt(1 + i * 0.1)
    lo = v - width
    hi = v + width
    print(f"+{i+1:<7} ${v:>11,.0f} ${lo:>11,.0f} ${hi:>11,.0f} ${2*width:>11,.0f}")

## 7. Forecast Fallback: Simple Exponential Smoothing

Used when statsmodels is unavailable or the series is too short (< 3 points). Maintains the same output schema.

In [ ]:
def exponential_smoothing_fallback(
    values: np.ndarray,
    periods: int = 6,
    alpha: float = 0.3,  # smoothing factor: higher = more weight to recent observations
) -> dict:
    """Simple exponential smoothing without trend component."""
    level = values[0]
    fitted_vals = [level]

    for v in values[1:]:
        level = alpha * v + (1 - alpha) * level
        fitted_vals.append(level)

    residuals = values - np.array(fitted_vals)
    rse = float(np.std(residuals)) if len(residuals) > 1 else 0.0
    ci_mult = 1.96

    forecast_vals = []
    ci_lower = []
    ci_upper = []
    for i in range(periods):
        forecast_vals.append(round(float(level), 2))
        width = rse * ci_mult * np.sqrt(1 + i * 0.1)
        ci_lower.append(round(float(level - width), 2))
        ci_upper.append(round(float(level + width), 2))

    return {
        "forecast": forecast_vals,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "fitted": [round(float(v), 2) for v in fitted_vals],
    }


fallback_result = exponential_smoothing_fallback(mrr_series, periods=6, alpha=0.3)
print("Fallback forecast (flat, no trend):")
for i, v in enumerate(fallback_result["forecast"], 1):
    print(f"  Period +{i}: ${v:>10,.0f}")